# Retail Customer Analytics with Apache Spark
### Spark SQL exploratory analysis & MLlib K-Means segmentation

**Course:** Business Intelligence and Data Engineering — MSc in Business Analytics, Athens University of Economics and Business

**Context.** A quick-commerce (Q-Commerce) company wishes to understand the purchasing behaviour of its customers. We analyse a dataset of 1,000,000 online retail transactions using Apache Spark's DataFrame API, Spark SQL and the MLlib machine-learning library.

**Structure of the notebook**
- **Step 0** — Environment set-up and Spark session
- **Step 1** — Data loading and schema definition
- **Part 1** — Exploratory analysis with Spark SQL (questions 1–9)
- **Part 2** — Customer segmentation with K-Means (MLlib)

## Step 0 — Environment set-up and Spark session

Every Spark application is coordinated by a **driver program**, which communicates with the cluster through a `SparkSession` object. The `SparkSession` is the single entry point to the DataFrame API, Spark SQL and MLlib.

In this notebook Spark runs in **local mode** (`local[*]`): the driver and the executors run on the same machine, and `*` instructs Spark to use as many worker threads as there are CPU cores. The same code would run unchanged on a cluster; only the `master` setting would differ.

> **Requirements:** Java 17 (or 21) and PySpark 3.x/4.x. If Java is not found, set the `JAVA_HOME` environment variable before starting Jupyter.

In [ ]:
# --- Imports used throughout the notebook ---
from pyspark.sql import SparkSession
from pyspark.sql import functions as F          # built-in SQL functions (sum, avg, col, ...)
from pyspark.sql.types import (StructType, StructField, LongType, IntegerType,
                               DoubleType, StringType, BooleanType, TimestampType)

# --- Create (or reuse, if it already exists) the Spark session ---
spark = (SparkSession.builder
         .master("local[*]")                    # local mode, one thread per CPU core
         .appName("retail-customer-analytics")  # name shown in the Spark UI
         .config("spark.driver.memory", "4g")   # enough memory for 1M rows
         # Apache Arrow: fast, columnar Spark -> pandas conversion (used by toPandas())
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")         # hide verbose INFO/WARN messages
print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

## Step 1 — Data loading

The dataset is a CSV file with a header line. Spark can either **infer** the schema (`inferSchema=True`) or receive it **explicitly**.

We define the schema explicitly, for two reasons:
1. **Performance** — schema inference requires an extra full pass over the file just to guess the data types.
2. **Correctness** — we control the type of every column (e.g. `Date` becomes a proper timestamp and `Discount_Applied` a boolean), instead of relying on Spark's guesses.

> Set `DATA_PATH` to the location of the CSV file on your machine.

In [ ]:
DATA_PATH = "data/retail-dataset.csv"   # <-- adjust if the file is stored elsewhere

# Explicit schema: one StructField(name, type, nullable) per column, in file order
schema = StructType([
    StructField("Transaction_ID",    LongType(),      False),
    StructField("Date",              TimestampType(), True),
    StructField("Customer_Name",     StringType(),    True),
    StructField("Product",           StringType(),    True),  # list stored as text, e.g. "['Milk', 'Bread']"
    StructField("Total_Items",       IntegerType(),   True),
    StructField("Total_Cost",        DoubleType(),    True),
    StructField("Payment_Method",    StringType(),    True),
    StructField("City",              StringType(),    True),
    StructField("Store_Type",        StringType(),    True),
    StructField("Discount_Applied",  BooleanType(),   True),
    StructField("Customer_Category", StringType(),    True),
    StructField("Season",            StringType(),    True),
    StructField("Promotion",         StringType(),    True),
])

# Reading is lazy: Spark only records *how* to read the file; no data is loaded yet
retail = (spark.read
          .option("header", "true")      # first line contains the column names
          .option("quote", '"')          # Product values contain commas inside quotes
          .schema(schema)
          .csv(DATA_PATH))

retail.printSchema()

Spark distinguishes between **transformations** (e.g. `read`, `select`, `filter`), which are *lazy* and only build an execution plan, and **actions** (e.g. `count`, `show`, `toPandas`), which trigger the actual computation. The cell below contains two actions; this is the moment the file is actually read.

Because the DataFrame will be queried many times in the following sections, we `cache()` it so that, after the first action, it is kept in memory instead of being re-read from disk each time.

In [ ]:
retail.cache()                                   # keep the data in memory after first use

print(f"Number of transactions: {retail.count():,}")  # action -> reads the file and fills the cache
print(f"Number of columns:      {len(retail.columns)}")

# Show the first five rows; toPandas() renders a nicer table in Jupyter
retail.limit(5).toPandas()